# Statistical analysis of lop algorithms implementation

## setup

In [1]:
!uv sync


Resolved 46 packages in 16ms
Checked 40 packages in 9ms


In [2]:
import ast
import itertools
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel, wilcoxon


### parameters

In [3]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "input").is_dir() and (PROJECT_ROOT.parent / "data" / "input").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "analysis" / "benchmark_best_known.py").is_file():
    raise FileNotFoundError(f"Could not locate project root from {Path.cwd()}")

PATH_TO_OUT = PROJECT_ROOT / "data" / "output"
PATH_TO_IN = PROJECT_ROOT / "data" / "input"

PATH_TO_BEST_KNOWN = PATH_TO_IN / "best_known.txt"
PATH_TO_INSTANCES = PATH_TO_IN / "instances"
PATH_TO_OUTPUT_IT_IMP = PATH_TO_OUT / "it_im_results.csv"
PATH_TO_OUTPUT_VND = PATH_TO_OUT / "lop_vnd_results.csv"
PATH_TO_OUTPUT_MEM_PARAM = PATH_TO_OUT / "meme_param_results.csv"
PATH_TO_OUTPUT_MEMETIC = PATH_TO_OUT / "meme_results.csv"
PATH_TO_OUTPUT_ILS = PATH_TO_OUT / "ils_results.csv"
PATH_TO_OUTPUT_ILS_PARAM = PATH_TO_OUT / "ils_param_results.csv"    

PATH_TO_FIGS = PATH_TO_OUT / "figs"

PATH_BENCHMARK = PROJECT_ROOT / "analysis" / "benchmark_best_known.py"

bench_out = {
    "it_imp": PATH_TO_OUTPUT_IT_IMP,
    "vnd": PATH_TO_OUTPUT_VND,
    "meme_param": PATH_TO_OUTPUT_MEM_PARAM,
    "meme": PATH_TO_OUTPUT_MEMETIC,
    "ils": PATH_TO_OUTPUT_ILS,
    "ils_param": PATH_TO_OUTPUT_ILS_PARAM,
}

# convenience ordered list for loops
benches = list(bench_out.items())

# benchmark parameters
N_WORKERS = -1
N_RUNS = 1
# Guard to avoid benchmarks
RUN_BENCH = True


results_groups = ["cost", "elapsed_seconds", "solution"]

### functions

In [13]:
def ensure_benchmark(bench: str, outputs: Path, timeout:int = 0) -> None:
    if outputs.exists():
        print(f"{bench} benchmark already present. Skipping.")
        return
    elif (RUN_BENCH is False):
        print(f"{bench} CSV missing but RUN_BENCH is False — skipping benchmark run.")
        return
    

    if not PATH_BENCHMARK.exists():
        raise FileNotFoundError(f"Benchmark script not found: {PATH_BENCHMARK}")


    PATH_TO_OUT.mkdir(parents=True, exist_ok=True)

    cmd = [
        "uv",
        "run",
        str(PATH_BENCHMARK),
        "--bench",
        bench,
        "--instances-dir",
        str(PATH_TO_INSTANCES),
        "--best_known_file",
        str(PATH_TO_BEST_KNOWN),
        "--output",
        str(PATH_TO_OUT),
        "--runs",
        str(N_RUNS),
        "--workers",
        str(N_WORKERS),
        "--timeout",
        str(timeout)
    ]
    exit_code = subprocess.call(cmd, cwd=PROJECT_ROOT)
    if exit_code != 0:
        raise RuntimeError(f"Benchmark '{bench}' failed.")
    

def load_benchmark(bench: str) -> pd.DataFrame:
    ensure_benchmark(bench, bench_out[bench])
    df = pd.read_csv(bench_out[bench])
    non_results = df.columns.difference(results_groups)
    df = df.groupby(non_results)[results_groups].apply(lambda x:x)
    
    return df

In [10]:
def parse_neighborhoods(value: object) -> tuple[str, ...]:
    if isinstance(value, (list, tuple)):
        return tuple(str(item) for item in value)
    parsed = ast.literal_eval(str(value))
    if not isinstance(parsed, (list, tuple)):
        raise ValueError(f"Invalid neighborhoods value: {value}")
    return tuple(str(item) for item in parsed)


def run_pairwise_tests(gap_matrix: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for left, right in itertools.combinations(gap_matrix.columns, 2):
        paired = gap_matrix[[left, right]].dropna()
        if paired.empty:
            continue

        left_values = paired[left].to_numpy()
        right_values = paired[right].to_numpy()

        t_stat, t_p_value = ttest_rel(left_values, right_values, alternative="two-sided")

        diff = left_values - right_values
        if np.allclose(diff, 0.0):
            w_stat, w_p_value = 0.0, 1.0
        else:
            w_stat, w_p_value = wilcoxon(
                left_values,
                right_values,
                alternative="two-sided",
                zero_method="wilcox",
            )

        rows.append(
            {
                "algorithm_a": left,
                "algorithm_b": right,
                "n_instances": len(paired),
                "mean_gap_a": left_values.mean(),
                "mean_gap_b": right_values.mean(),
                "ttest_p_value": t_p_value,
                "wilcoxon_p_value": w_p_value,
                "significant_5pct_wilcoxon": w_p_value < 0.05,
            }
        )

    return pd.DataFrame(rows).sort_values("wilcoxon_p_value", ignore_index=True)


In [11]:
def algo_progression(csv_path: Path) -> pd.DataFrame:
    """Return the cost progression over time for each benchmark parameter combination.

    The returned table uses a MultiIndex on the rows made of the algorithm
    parameters plus the instance name, and one column per elapsed time. Each
    cell stores the best-so-far cost reached at that time.
    """
    df = pd.read_csv(csv_path).copy()

    if "elapsed_seconds" not in df.columns and "time_s" in df.columns:
        df["elapsed_seconds"] = df["time_s"]

    if "neighborhoods" in df.columns and "neighborhood" not in df.columns:
        df["neighborhood"] = df["neighborhoods"]

    if "neighborhood" in df.columns:
        df["neighborhood"] = df["neighborhood"].apply(parse_neighborhoods)

    required_columns = {"instance", "cost", "elapsed_seconds"}
    missing_columns = sorted(required_columns - set(df.columns))
    if missing_columns:
        raise ValueError(
            f"{csv_path} is missing required columns: {', '.join(missing_columns)}"
        )

    metric_columns = {"instance", "best_known_cost", "cost", "elapsed_seconds", "time_s", "solution"}
    parameter_columns = [column for column in df.columns if column not in metric_columns]
    if not parameter_columns:
        raise ValueError(f"{csv_path} does not contain any algorithm parameter columns")

    df = df.sort_values(parameter_columns + ["instance", "elapsed_seconds"]).copy()
    df["elapsed_seconds"] = df["elapsed_seconds"].astype(float).round(6)
    df["best_cost_so_far"] = df.groupby(parameter_columns + ["instance"])["cost"].cummax()

    progression = (
        df.pivot_table(
            index=parameter_columns + ["instance"],
            columns="elapsed_seconds",
            values="best_cost_so_far",
            aggfunc="last",
        )
        .sort_index(axis=1)
    )
    progression.columns.name = "elapsed_seconds"
    return progression


In [12]:
def analyze_benchmark(bench: str, csv_path: Path) -> None:
    print(f"\n--- Processing benchmark: {bench} ---")
    # Run the benchmark only if the expected CSV output is missing
    ensure_benchmark(bench, [csv_path])

    # If benchmark run was skipped and CSV still missing, avoid crashing
    if not csv_path.exists():
        print(f"No CSV found for {bench} at {csv_path}; skipping analysis.")
        return

    # Try the strict loader first; fallback to flexible parsing for param-sweep CSVs
    try:
        df = load_results(csv_path)
    except ValueError as exc:
        print(f"load_results failed: {exc}; using flexible loader for {csv_path}")
        df = pd.read_csv(csv_path).copy()
        df = normalize_columns(df)
        # time column fallback
        if "time_s" not in df.columns and "elapsed_seconds" in df.columns:
            df["time_s"] = df["elapsed_seconds"]
        # neighborhood parsing fallback
        if "neighborhoods" in df.columns:
            df["neighborhood_list"] = df["neighborhoods"].map(parse_neighborhoods)
        elif "neighborhood" in df.columns:
            df["neighborhood_list"] = df["neighborhood"].map(lambda v: (v,))
        else:
            # no neighborhood info available -> empty tuple
            df["neighborhood_list"] = [() for _ in range(len(df))]
        # best-known and cost required
        if "best_known_cost" not in df.columns or "cost" not in df.columns:
            raise ValueError(f"Cannot analyze {csv_path}: missing cost/best-known columns")
        df["gap_pct"] = (df["best_known_cost"] - df["cost"]) / df["best_known_cost"] * 100.0

    if df.empty:
        print(f"No rows for {bench} in {csv_path}")
        return

    # Ensure a consistent `algorithm` column for grouping/plotting
    if "algorithm" not in df.columns:
        if "neighborhood_list" in df.columns:
            def _alg_from_neigh(n):
                try:
                    return " -> ".join(n) if len(n) > 1 else n[0]
                except Exception:
                    return str(n)
            df["algorithm"] = df["neighborhood_list"].map(_alg_from_neigh)
        else:
            df["algorithm"] = (
                df.get("sol_start", "").astype(str)
                + " | "
                + df.get("pivot", "").astype(str)
            )

    summary = (
        df.groupby(["algorithm"], as_index=False)
        .agg(
            avg_gap_pct=("gap_pct", "mean"),
            std_gap_pct=("gap_pct", "std"),
            total_time_s=("time_s", "sum"),
        )
        .sort_values("avg_gap_pct", ignore_index=True)
    )

    summary.to_csv(PATH_TO_OUT / f"{bench}_summary_stats.csv", index=False)

    # Build gap matrix; if duplicate instance/algorithm rows exist, aggregate by mean first
    try:
        gap_matrix = df.pivot(index="instance", columns="algorithm", values="gap_pct")
    except Exception as exc:
        print(f"pivot failed ({exc}); aggregating duplicates by mean before pivot")
        gap_matrix = df.groupby(["instance", "algorithm"], as_index=False)["gap_pct"].mean().pivot(index="instance", columns="algorithm", values="gap_pct")

    if gap_matrix.shape[1] >= 2:
        pairwise = run_pairwise_tests(gap_matrix)
        pairwise.to_csv(PATH_TO_OUT / f"{bench}_pairwise_tests.csv", index=False)

    display(summary)

    PATH_TO_FIGS.mkdir(parents=True, exist_ok=True)
    order = summary["algorithm"].tolist()

    fig, ax = plt.subplots(figsize=(6, 6), constrained_layout=True)
    ax.boxplot(
        [df.loc[df["algorithm"] == algo, "gap_pct"] for algo in order],
        tick_labels=order,
        showmeans=True,
    )
    ax.set_title(f"{bench}: instance-wise deviation distribution")
    ax.set_ylabel("Deviation from best-known (%)")
    ax.set_xlabel("Algorithm")
    ax.tick_params(axis="x", rotation=45)

    fig.savefig(PATH_TO_FIGS / f"{bench}_boxplot.svg", format="svg")
    plt.show()

print("Note: RUN_BENCH flag controls whether missing CSVs will be computed. It is currently:", RUN_BENCH)

Note: RUN_BENCH flag controls whether missing CSVs will be computed. It is currently: True


## Iterative improvement
Run benchmarks of iterative improvement 'it_imp'.

In [14]:
load_benchmark("it_imp"),
ensure_benchmark("it_imp", [PATH_TO_OUTPUT_IT_IMP])

df_it_imp = load_results(PATH_TO_OUTPUT_IT_IMP)
df_it_imp = df_it_imp[df_it_imp["neighborhood_list"].str.len() == 1].copy()
if df_it_imp.empty:
    raise ValueError("No iterative improvement rows found in it_im_results.csv.")

df_it_imp["neighborhood"] = df_it_imp["neighborhood_list"].str[0]
df_it_imp["algorithm"] = (
    df_it_imp["sol_start"] + " | " + df_it_imp["pivot"] + " | " + df_it_imp["neighborhood"]
)

it_summary = (
    df_it_imp.groupby(["sol_start", "pivot", "neighborhood", "algorithm"], as_index=False)
    .agg(
        avg_gap_pct=("gap_pct", "mean"),
        std_gap_pct=("gap_pct", "std"),
        total_time_s=("time_s", "sum"),
    )
    .sort_values(["avg_gap_pct", "total_time_s"], ignore_index=True)
)
it_summary.to_csv(PATH_TO_OUT / "it_im_summary_stats.csv", index=False)

it_gap_matrix = df_it_imp.pivot(index="instance", columns="algorithm", values="gap_pct")
it_pairwise_tests = run_pairwise_tests(it_gap_matrix)
it_pairwise_tests.to_csv(PATH_TO_OUT / "it_im_pairwise_tests.csv", index=False)

display(it_summary)
display(it_pairwise_tests.head(20))

plt.style.use("ggplot")  # emulate R style
PATH_TO_FIGS.mkdir(parents=True, exist_ok=True)

it_plot = it_summary.sort_values("avg_gap_pct")
it_order = it_plot["algorithm"].tolist()

fig_it_summary, axes = plt.subplots(figsize=(18, 6), constrained_layout=True)
axes.bar(it_plot["algorithm"], it_plot["total_time_s"], color="#457b9d")
axes.set_title("Iterative improvement: average time")
axes.set_ylabel("Total time across all instances (s)")
axes.set_xlabel("Algorithm")
axes.tick_params(axis="x", rotation=65)

fig_it_summary.savefig(PATH_TO_FIGS / "it_imp_summary.svg", format="svg")
plt.show()

fig_it_box, ax = plt.subplots(figsize=(16, 6), constrained_layout=True)
ax.boxplot(
    [df_it_imp.loc[df_it_imp["algorithm"] == algo, "gap_pct"] for algo in it_order],
    tick_labels=it_order,
    showmeans=True,
)
ax.set_title("Iterative improvement: instance-wise deviation distribution")
ax.set_ylabel("Deviation from best-known (%)")
ax.set_xlabel("Algorithm")
ax.tick_params(axis="x", rotation=65)
fig_it_box.savefig(PATH_TO_FIGS / "it_imp_boxplot.svg", format="svg")
plt.show()


it_imp benchmark already present. Skipping.


ValueError: Grouper and axis must be same length

## Variable neighborhood descent (VND) algorithm

In [ ]:
ensure_benchmark("vnd", [PATH_TO_OUTPUT_VND])

df_vnd = load_results(PATH_TO_OUTPUT_VND)
df_vnd = df_vnd[df_vnd["neighborhood_list"].str.len() > 1].copy()
if df_vnd.empty:
    raise ValueError("No VND rows found in lop_vnd_results.csv.")

df_vnd["algorithm"] = df_vnd["neighborhood_list"].map(lambda n: " -> ".join(n))

vnd_summary = (
    df_vnd.groupby(["algorithm"], as_index=False)
    .agg(
        avg_gap_pct=("gap_pct", "mean"),
        std_gap_pct=("gap_pct", "std"),
        total_time_s=("time_s", "sum"),
    )
    .sort_values("avg_gap_pct", ignore_index=True)
)
vnd_summary.to_csv(PATH_TO_OUT / "vnd_summary_stats.csv", index=False)

vnd_gap_matrix = df_vnd.pivot(index="instance", columns="algorithm", values="gap_pct")
if vnd_gap_matrix.shape[1] != 2:
    raise ValueError(f"Expected exactly 2 VND algorithms, got {vnd_gap_matrix.shape[1]}")

vnd_a, vnd_b = vnd_gap_matrix.columns
vnd_paired = vnd_gap_matrix[[vnd_a, vnd_b]].dropna()

vnd_t_stat, vnd_t_p_value = ttest_rel(vnd_paired[vnd_a], vnd_paired[vnd_b], alternative="two-sided")
vnd_diff = vnd_paired[vnd_a].to_numpy() - vnd_paired[vnd_b].to_numpy()
if np.allclose(vnd_diff, 0.0):
    vnd_w_stat, vnd_w_p_value = 0.0, 1.0
else:
    vnd_w_stat, vnd_w_p_value = wilcoxon(
        vnd_paired[vnd_a],
        vnd_paired[vnd_b],
        alternative="two-sided",
        zero_method="wilcox",
    )

vnd_test = pd.DataFrame(
    [
        {
            "algorithm_a": vnd_a,
            "algorithm_b": vnd_b,
            "n_instances": len(vnd_paired),
            "mean_gap_a": vnd_paired[vnd_a].mean(),
            "mean_gap_b": vnd_paired[vnd_b].mean(),
            "ttest_p_value": vnd_t_p_value,
            "wilcoxon_p_value": vnd_w_p_value,
            "significant_5pct_wilcoxon": vnd_w_p_value < 0.05,
        }
    ]
)

vnd_test.to_csv(PATH_TO_OUT / "vnd_stat_test.csv", index=False)

display(vnd_summary)
display(vnd_test)

PATH_TO_FIGS.mkdir(parents=True, exist_ok=True)

vnd_order = vnd_summary["algorithm"].tolist()
fig_vnd, axes = plt.subplots(figsize=(5, 5), constrained_layout=True)


axes.boxplot(
    [df_vnd.loc[df_vnd["algorithm"] == algo, "gap_pct"] for algo in vnd_order],
    tick_labels=vnd_order,
    showmeans=True,
)
axes.set_title("VND: instance-wise deviation distribution")
axes.set_ylabel("Deviation from best-known (%)")
axes.set_xlabel("Neighborhood order")
axes.tick_params(axis="x", rotation=20)

fig_vnd.savefig(PATH_TO_FIGS / "vnd_summary.svg", format="svg")
plt.show()


## iterative local search (ILS) algorithm

### parameters
Run 'meme_param' benchmark and compare the results. This allow to visualize the effect of the parameters on the solution quality. Only on instances is used to reduce computation time.

In [ ]:
analyze_benchmark("ils_param", bench_out["ils_param"])

### instances
Run 'memetic' benchmark on all instances of size 150 and show the deviation from best known solution.

In [ ]:
analyze_benchmark("ils", bench_out["ils"])

## Memetic algorithm
First make the 'meme_param' benchmark, to determine the best parameters for the memetic algorithm. Then run the memetic algorithm with the `memetic` benchmark on all instances of size 150 and analyze the results.

### Parameters
Run 'meme_param' benchmark and compare the results. 

In [ ]:
analyze_benchmark("meme_param", bench_out["meme_param"])

### instances
Run 'memetic' benchmark to measure the deviation from best known on all instances of size 150.

In [ ]:

analyze_benchmark("meme", bench_out["meme"])